# Validation: lifetime measurements (RDDS and DSAM)

The plunger's decay curves and Bateman's equations against their closed forms, the deorientation in vacuum against Radeck et al.'s Table I, the recoil transport against TRIM, the DSAM attenuation against its exact result for stopping proportional to speed, and DSAM line shapes against APCAD's and AhKin's examples (shape only, not a lifetime fit); the DDCM against NaPyTau on physim's own simulated plunger spectra, τ recovered from simulated runs of known τ (closure), and the three published experiments (⁹⁸Ru, ⁵⁰Cr, ¹⁴³Sm) simulated from the examples made after the papers and analysed as the papers did (`physim.nuclear.lifetime_reproduction`). The two line-shape comparisons and the ¹⁴³Sm reproduction fail their tolerances; why is on the register page `docs/physics-register/lifetimes.md`.

Each check is defined in `physim.nuclear.validation`; the same checks run in the test suite (`tests/python/test_nuclear_validation.py`), and the physics register may show ✅ only for checks that pass. 🟡 means the reference has been requested but not produced yet (see `tests/reference/nuclear/pending/`).

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Markdown
from physim.nuclear import validation

capabilities = ['Lifetimes: recoil distance (RDDS)', 'Lifetimes: Doppler-shift attenuation (DSAM)']
results = [r for c in capabilities for r in validation.run(capability=c)]
Markdown(validation.report(results))

## Comparisons

Red dots are physim, open squares the reference.

In [ ]:
shown = [r for r in results if r.plot]
fig, axes = plt.subplots(len(shown), 1, figsize=(6.5, 3.2 * len(shown)), squeeze=False)
for ax, r in zip(axes[:, 0], shown):
    validation.plot(r, ax=ax)
fig.tight_layout()

## Deorientation in vacuum, ⁹⁸Ru 2⁺₁

Radeck et al.'s Table I parameters: the attenuation of the unshifted γ rays, G_k at the stopper, and of the shifted ones, G̃_k averaged over the decays in flight (their Eq. 7), against the target-to-stopper distance.

In [ ]:
r = next(x for x in results if x.name == 'deorientation_ru98')
fig, ax = plt.subplots(figsize=(6.5, 3.2))
for k, colour in ((2, '#d62728'), (4, '#1f77b4')):
    c = r.plot['curves'][k]
    ax.plot(r.plot['x'], c['unshifted'], color=colour, label=f'G{k} (unshifted)')
    ax.plot(r.plot['x'], c['shifted'], '--', color=colour, label=f'G̃{k} (shifted)')
ax.set_xlabel('distance (µm)')
ax.set_ylabel('attenuation')
ax.legend(fontsize=8)
fig.tight_layout()

## The ⁸³Y line shape

AhKin's example at its τ = 0.1313 ps: the measured line, AhKin's best line and physim's with AhKin's inputs (its stopping table, cascade, detector and resolution).

In [ ]:
r = next(x for x in results if x.name == 'dsam_lineshape_vs_ahkin')
fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.step(r.plot['x'], r.plot['reference'], where='mid', color='#444444', label='measured')
ax.plot(r.plot['x'], r.plot['ahkin'], color='#1f77b4', label='AhKin')
ax.plot(r.plot['x'], r.plot['physim'], color='#d62728', label='physim')
ax.set_xlabel('energy (keV)')
ax.set_ylabel('counts per channel')
ax.legend(fontsize=8)
fig.tight_layout()

## ⁹⁸Ru after Coulex (Radeck et al. 2012)

The example simulated at its 11 distances, particle–γ with the ²⁴Mg recoils. The unshifted intensity per ring against distance, divided by its angular distribution W_U(d) (the deorientation correction), and the decay-curve fit; and τ with and without the correction against the paper's 8.36(29) ps.

In [ ]:
import numpy as np
from physim.nuclear import lifetime_reproduction as lr

ru = lr.ru98()
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
for (ring, r), colour in zip(sorted(ru['rings'].items()), ('#d62728', '#1f77b4')):
    y = np.array(r['unshifted']) / np.array(r['w_unshifted'])
    e = np.array(r['err_unshifted']) / np.array(r['w_unshifted'])
    ax[0].errorbar(ru['distances_um'], y, e, fmt='o', ms=3, color=colour, label=f'{ring:g}°')
    ax[0].plot(ru['distances_um'], r['model'], color=colour, lw=0.8)
ax[0].set_xscale('log'); ax[0].set_yscale('log')
ax[0].set_xlabel('distance (µm)'); ax[0].set_ylabel('unshifted / W_U'); ax[0].legend(fontsize=8)
labels = ['paper', 'physim, corrected', 'physim, uncorrected']
vals = [ru['paper_tau_ps'], ru['tau_ps'], ru['uncorrected_ps']]
errs = [ru['paper_sigma_ps'], ru['stat_ps'], 0]
ax[1].errorbar(range(3), vals, errs, fmt='s', color='#444444')
ax[1].set_xticks(range(3)); ax[1].set_xticklabels(labels, fontsize=8); ax[1].set_ylabel('τ(2⁺₁) (ps)')
fig.tight_layout()
print(ru['detail'])

## ⁵⁰Cr after fusion-evaporation, three-foil plunger (Beckers et al. 2022)

τ(x) of the 2⁺₁ from the degrader (t₁) and stopper (t₂) parts of the γγ DDCM with direct gates (their Eq. 13), against the effective distance, with the paper's value and the other published values.

In [ ]:
cr = lr.cr50()
fig, ax = plt.subplots(figsize=(6.5, 3.2))
for part, colour in (('t1', '#d62728'), ('t2', '#1f77b4')):
    pts = [p for p in cr['per_distance'][part] if p['used']]
    ax.errorbar([p['distance_um'] for p in pts], [p['tau_ps'] for p in pts],
                [p['error_ps'] for p in pts], fmt='o', ms=3, color=colour, label=f'{part} part')
for method, tau, unc, _src in cr['states']['2+1']['published']:
    ax.axhspan(tau - unc, tau + unc, alpha=0.12, color='#444444')
ax.axhline(cr['states']['2+1']['tau_ps'], color='#2ca02c', label='physim')
ax.set_xscale('log'); ax.set_xlabel('effective distance (µm)'); ax.set_ylabel('τ(x), 2⁺₁ (ps)')
ax.legend(fontsize=8); fig.tight_layout()
print(cr['detail'])

## ¹⁴³Sm after fusion-evaporation, DSAM on Au (Rajbanshi et al. 2018)

The lifetimes fitted with SRIM's stopping (as the paper) from line shapes simulated with physim's, against the paper's Table I (the two upper limits marked), and one simulated line with the fit.

In [ ]:
sm = lr.sm143()
fig, ax = plt.subplots(1, 2, figsize=(9, 3.4))
order = sm['order']
x = np.arange(len(order))
paper = [sm['levels'][k]['paper_tau_ps'] for k in order]
ax[0].plot(x, paper, 's', mfc='none', color='#444444', label='paper (Table I)')
ax[0].errorbar(x, [sm['levels'][k]['tau_ps'] for k in order], [sm['levels'][k]['stat_ps'] for k in order],
               fmt='o', ms=3, color='#d62728', label='physim, SRIM kernel')
ax[0].set_xticks(x); ax[0].set_xticklabels([f"{sm['levels'][k]['e_gamma_kev']:.0f}" for k in order],
                                           rotation=45, fontsize=7)
ax[0].set_xlabel('γ ray (keV)'); ax[0].set_ylabel('τ (ps)'); ax[0].legend(fontsize=8)
lab = '49/2- (DB I)'
edges = sm['spectra'][lab]['edges']
c = 0.5 * (edges[1:] + edges[:-1])
for th, colour in zip(sm['angles'], ('#d62728', '#1f77b4')):
    ax[1].step(c, sm['spectra'][lab]['counts'][th], where='mid', color=colour, label=f'{th:g}°')
ax[1].set_xlabel('energy (keV)'); ax[1].set_ylabel('counts'); ax[1].set_title('444.8 keV', fontsize=9)
ax[1].legend(fontsize=8); fig.tight_layout()
print(sm['detail'])